---
Preliminary analysis of correlations between grain boundary character and hardness values
---

In [4]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
from GPy.models import GPRegression
from GPy.kern import RBF, Matern32, Matern52
from GPy.mappings import Constant, Linear
from scripts import PCAGaussianProcess, BasisGaussianProcess


## Opening the data file and analyzing columns and data types
* Use `pandas` to read `data/Berkovich nanoindentation vs Block Parameters.xlsx` (only sheet `H_(E-2)`)
* Description of the excel file:
  * First column: index 
  * Second column: testId
  * 3rd column: Hardness (GPa)
  * 4th column: Distance (um) - distance of the indentation from the grain boundary
  * 5th column: Misorientation (deg)
  * 6,7,8th columns: Euler angles (deg) - Euler angles of the indented grain
  * 10,11,12th columns: Euler angles (deg) - Euler angles of the neighboring grain on the other side of the grain boundary
* numerical data starts from the third row of the excel file
* No need to read other rows or columns
* missing data is ignored


In [7]:
file_path = "data/Berkovich nanoindentation vs Block Parameters.xlsx"
df = pd.read_excel(
    file_path,
    sheet_name="H_(E-2)",
    skiprows=2,
    header=None,
    usecols=[0, 1, 2, 3, 4, 5, 6, 7, 9, 10, 11],
    names=[
        "Index",
        "Test",
        "Hardness (GPa)",
        "Distance (um)",
        "Misorientation (degrees)",
        "Euler angle 1 (indent)",
        "Euler angle 2 (indent)",
        "Euler angle 3 (indent)",
        "Euler angle 1 (neighbor)",
        "Euler angle 2 (neighbor)",
        "Euler angle 3 (neighbor)",
    ],
).dropna()

df.describe()
df.head()

,Index,Test,Hardness (GPa),Distance (um),Misorientation (degrees),Euler angle 1 (indent),Euler angle 2 (indent),Euler angle 3 (indent),Euler angle 1 (neighbor),Euler angle 2 (neighbor),Euler angle 3 (neighbor)
0,1.0,2_214,5.05322,0.52,9.16,33.34,11.57,8.99,78.67,7.54,52.86
1,2.0,2_62,5.30000,0.35,17.19,144.41,9.60,31.50,186.55,21.53,87.51
2,3.0,2_28,4.60000,0.40,13.78,44.55,2.69,19.63,60.93,4.65,20.10
3,4.0,2_98,5.00000,0.34,12.97,204.64,25.14,73.44,222.25,18.38,66.66
4,5.0,2_79,4.50000,0.41,27.76,207.77,9.53,49.35,284.88,25.32,47.08


In [8]:
inputs, outputs = df.iloc[:, 3:9].values, df.iloc[:, 2].values.reshape(-1,1)

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split


indx = np.arange(inputs.shape[0])

indx_train, indx_test, inputs_train, inputs_test, outputs_train, outputs_test = train_test_split(
    indx,
    inputs, 
    outputs, 
    test_size=0.2, 
    random_state=42
)

input_scaler = MinMaxScaler()
output_scaler = MinMaxScaler()

Xtrain = input_scaler.fit_transform(inputs_train)
Xtest = input_scaler.transform(inputs_test)
Ytrain = output_scaler.fit_transform(outputs_train)
Ytest = output_scaler.transform(outputs_test)



In [9]:
kernel = Matern32(input_dim=6, ARD=True)
mean = Linear(input_dim=6, output_dim=1)
model = GPRegression(
    Xtrain, 
    Ytrain, 
    kernel=kernel, 
    mean_function=mean
)
model.optimize_restarts(
    num_restarts=10, 
    verbose=True
)

Optimization restart 1/10, f = -61.24870807036859
Optimization restart 2/10, f = -61.2486166716041
Optimization restart 3/10, f = -61.24783486075647
Optimization restart 4/10, f = -61.410451262463
Optimization restart 5/10, f = -61.15598881646747
Optimization restart 6/10, f = -61.155916940649774
Optimization restart 7/10, f = -61.248756698418305
Optimization restart 8/10, f = -61.24879329677326
Optimization restart 9/10, f = -61.24877970490334
Optimization restart 10/10, f = -61.24801729062878


In [ ]:
ytest_pred, ytest_var = model.predict(Xtest)

# scaling back the predictions to the original scale
ytest_pred_original = output_scaler.inverse_transform(ytest_pred)
output_range = output_scaler.data_max_ - output_scaler.data_min_
ytest_var_original = ytest_var * (output_range ** 2)
ytest_std_original = np.sqrt(ytest_var_original)



In [ ]:
# Parity plots between predicted and actual values

plt.figure(figsize=(10, 6))
plt.scatter(outputs_test, ytest_pred_original, c=df['Distance (um)'].values, label='Predicted vs Actual')
plt.plot([outputs_test.min(), outputs_test.max()], [outputs_test.min(), outputs_test.max()], 'k--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Hardness (GPa)')
plt.ylabel('Predicted Hardness (GPa)')
plt.title('Parity Plot: Predicted vs Actual Hardness')
plt.legend()
plt.grid()
